# CUSTOMER INTELLIGENCE SUPPORT COPILOT

## Project Overview

The Customer Intelligence Support Copilot is a Python-based project designed to understand customer behaviour and improve customer support.

It performs customer segmentation, purchase analysis, support ticket classification, and refund policy assistance.

## Project Objectives

1. To analyse customer purchase behaviour.
2. To identify different customer segments using RFM analysis.
3. To predict inactive and at-risk customers.
4. To classify customer support tickets.
5. To retrieve refund information from policy documents.
6. To provide automated customer support assistance.

## Technologies Used

- Python
- Pandas
- NumPy
- Scikit-learn
- SQLite
- Jupyter Notebook


# PART A – DATA PREPARATION AND CLEANING

## Step 1: Import Libraries and Load Datasets

The aim of this step is to import the required Python libraries and load the raw customer, product and order datasets for analysis.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

# Set project directory
BASE_DIR = Path.cwd()

# Define raw data folder
RAW_DIR = BASE_DIR / "data" / "raw"

# Load datasets
customers = pd.read_csv(RAW_DIR / "customers.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
orders = pd.read_csv(RAW_DIR / "orders.csv")

# Display dataset information
print("CUSTOMER DATA")
display(customers.head())

print("\nPRODUCT DATA")
display(products.head())

print("\nORDER DATA")
display(orders.head())

print("\nAll datasets loaded successfully!")

CUSTOMER DATA


,customer_id,name,age,gender,city
0,1001,Customer_1,56.0,Male,Trichy
1,1002,Customer_2,46.0,Female,Chennai
2,1003,Customer_3,32.0,Female,Madurai
3,1004,Customer_4,25.0,Female,Coimbatore
4,1005,Customer_5,38.0,Male,Sivakasi



PRODUCT DATA


,product_id,product_name,category,price_inr
0,201,Product_1,Beverages,305
1,202,Product_2,Fruits,116
2,203,Product_3,Snacks,303
3,204,Product_4,Fruits,386
4,205,Product_5,Snacks,467



ORDER DATA


,order_id,customer_id,product_id,order_ts,status,quantity,discount_pct,order_amount,delivery_minutes
0,5001,1006,205,2026-01-01,Completed,2,5,887.3,40
1,5002,1099,206,2026-01-02,Cancelled,4,20,1510.4,89
2,5003,1037,219,2026-01-03,Completed,3,10,175.5,23
3,5004,1024,208,2026-01-04,Cancelled,4,5,1227.4,62
4,5005,1093,216,2026-01-05,Completed,1,20,191.2,57



All datasets loaded successfully!


## Step 2: Data Quality Audit

Data quality audit is used to identify missing values, duplicate records, and data types in the customer, product, and order datasets.

It helps us understand the quality of raw data before cleaning.

In [2]:
# DATA QUALITY AUDIT

datasets = {
    "Customers": customers,
    "Products": products,
    "Orders": orders
}

for name, df in datasets.items():

    print("\n" + "=" * 45)
    print(name.upper(), "DATA AUDIT")
    print("=" * 45)

    # Number of rows and columns
    print("Total Rows:", df.shape[0])
    print("Total Columns:", df.shape[1])

    # Missing values
    print("\nMissing Values:")
    print(df.isnull().sum())

    # Duplicate records
    print("\nDuplicate Records:", df.duplicated().sum())

    # Data types
    print("\nColumn Data Types:")
    print(df.dtypes)

    print("\nAudit completed for", name)

print("\nALL DATASETS AUDITED SUCCESSFULLY!")


CUSTOMERS DATA AUDIT
Total Rows: 100
Total Columns: 5

Missing Values:
customer_id    0
name           0
age            1
gender         0
city           1
dtype: int64

Duplicate Records: 0

Column Data Types:
customer_id      int64
name               str
age            float64
gender             str
city               str
dtype: object

Audit completed for Customers

PRODUCTS DATA AUDIT
Total Rows: 20
Total Columns: 4

Missing Values:
product_id      0
product_name    0
category        0
price_inr       0
dtype: int64

Duplicate Records: 0

Column Data Types:
product_id      int64
product_name      str
category          str
price_inr       int64
dtype: object

Audit completed for Products

ORDERS DATA AUDIT
Total Rows: 300
Total Columns: 9

Missing Values:
order_id            0
customer_id         0
product_id          0
order_ts            0
status              0
quantity            0
discount_pct        0
order_amount        0
delivery_minutes    0
dtype: int64

Duplicate Records:

## Step 3: Data Cleaning and Validation

Data cleaning is the process of removing duplicate records, handling missing values, and correcting invalid data.

In this step, customer, product, and order datasets are cleaned and prepared for further analysis.

In [3]:
# STEP 3: DATA CLEANING

# Create copies to preserve original data
clean_customers = customers.copy()
clean_products = products.copy()
clean_orders = orders.copy()

# 1. Remove duplicate records
clean_customers = clean_customers.drop_duplicates()
clean_products = clean_products.drop_duplicates()
clean_orders = clean_orders.drop_duplicates()

# 2. Clean customer data
if "name" in clean_customers.columns:
    clean_customers["name"] = clean_customers["name"].fillna("Unknown")

if "age" in clean_customers.columns:
    clean_customers["age"] = clean_customers["age"].fillna(
        clean_customers["age"].median()
    )

if "city" in clean_customers.columns:
    clean_customers["city"] = clean_customers["city"].fillna("Unknown")

# 3. Clean product data
if "category" in clean_products.columns:
    clean_products["category"] = clean_products["category"].fillna("Unknown")

# 4. Clean order data
if "quantity" in clean_orders.columns:
    clean_orders["quantity"] = clean_orders["quantity"].fillna(1)

# 5. Display cleaned dataset information
print("CUSTOMER DATA:", clean_customers.shape)
print("PRODUCT DATA:", clean_products.shape)
print("ORDER DATA:", clean_orders.shape)

# 6. Check remaining missing values
print("\nRemaining Missing Values:")
print("Customers:", clean_customers.isnull().sum().sum())
print("Products:", clean_products.isnull().sum().sum())
print("Orders:", clean_orders.isnull().sum().sum())

# 7. Save cleaned datasets
CLEAN_DIR = BASE_DIR / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

clean_customers.to_csv(CLEAN_DIR / "customers_clean.csv", index=False)
clean_products.to_csv(CLEAN_DIR / "products_clean.csv", index=False)
clean_orders.to_csv(CLEAN_DIR / "orders_clean.csv", index=False)

print("\nAll datasets cleaned and saved successfully!")

CUSTOMER DATA: (100, 5)
PRODUCT DATA: (20, 4)
ORDER DATA: (300, 9)

Remaining Missing Values:
Customers: 0
Products: 0
Orders: 0

All datasets cleaned and saved successfully!


## Step 4: Store Cleaned Data in SQLite Database

SQLite is a lightweight database used to store and manage structured data.

In this step, cleaned customer, product, and order datasets are stored as tables in an SQLite database.

In [4]:
# STEP 4: STORE DATA IN SQLITE DATABASE

import sqlite3

# Create database connection
DB_PATH = BASE_DIR / "data" / "instamart.db"

connection = sqlite3.connect(DB_PATH)

# Store cleaned datasets as database tables
clean_customers.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

clean_products.to_sql(
    "products",
    connection,
    if_exists="replace",
    index=False
)

clean_orders.to_sql(
    "orders",
    connection,
    if_exists="replace",
    index=False
)

# Display available tables
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    connection
)

print("DATABASE TABLES:")
display(tables)

# Check number of records
for table in ["customers", "products", "orders"]:
    result = pd.read_sql_query(
        f"SELECT COUNT(*) AS Total_Records FROM {table}",
        connection
    )
    print(f"\n{table.upper()} RECORD COUNT:")
    display(result)

connection.close()

print("\nAll cleaned datasets stored successfully in SQLite!")

DATABASE TABLES:


,name
0,customers
1,products
2,orders



CUSTOMERS RECORD COUNT:


,Total_Records
0,100



PRODUCTS RECORD COUNT:


,Total_Records
0,20



ORDERS RECORD COUNT:


,Total_Records
0,300



All cleaned datasets stored successfully in SQLite!


# PART B – CUSTOMER INTELLIGENCE AND SEGMENTATION

## Aim
To analyse customer purchasing behaviour using RFM analysis, customer segmentation, and machine learning techniques.

## Objectives
1. Calculate Recency, Frequency and Monetary values.
2. Identify customer segments.
3. Apply K-Means clustering.
4. Predict inactive and at-risk customers.

In [5]:
# =====================================================
# PART B - CUSTOMER INTELLIGENCE
# STEP 1 - RUN ALL ANALYSIS
# =====================================================

import sys
from pathlib import Path

# Connect notebook with src folder
PROJECT_DIR = Path.cwd()
SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Import customer intelligence module
import customer_intelligence as ci

# STEP 1: RFM Analysis
rfm = ci.rfm.copy()

print("\nRFM ANALYSIS COMPLETED!")
display(rfm.head(10))


# STEP 2: Customer Segmentation
segmented_customers = ci.customer_segmentation(rfm)

print("\nCUSTOMER SEGMENTATION COMPLETED!")
display(segmented_customers.head(10))


# STEP 3: K-Means Clustering
clustered_customers = ci.perform_kmeans(
    segmented_customers
)

print("\nK-MEANS CLUSTERING COMPLETED!")
display(clustered_customers.head(10))


# STEP 4: Customer Lapse Prediction
lapse_results = ci.predict_customer_lapse(
    clustered_customers
)

print("\nCUSTOMER LAPSE PREDICTION COMPLETED!")
display(lapse_results.head(10))


print("\nALL PART B ANALYSIS COMPLETED SUCCESSFULLY!")

Cleaned order data loaded successfully!
Completed orders: 229

========== RFM ANALYSIS ==========
   customer_id  Recency  Frequency  Monetary
0         1001      136          3    649.45
1         1002      103          2   1433.20
2         1004      134          1    601.80
3         1005        1          1   2218.25
4         1006      300          1    887.30
5         1007       10          3   2467.70
6         1008       98          3   1256.15
7         1009       47          3   1100.55
8         1010       83          3   1660.00
9         1012       37          4   2455.30

Total customers analysed: 83

RFM analysis completed successfully!
RFM file saved in: C:\Users\acer\OneDrive\Desktop\Customer_Intelligence_Support_Copilot\outputs

RFM ANALYSIS COMPLETED!


,customer_id,Recency,Frequency,Monetary
0,1001,136,3,649.45
1,1002,103,2,1433.20
2,1004,134,1,601.80
3,1005,1,1,2218.25
4,1006,300,1,887.30
5,1007,10,3,2467.70
6,1008,98,3,1256.15
7,1009,47,3,1100.55
8,1010,83,3,1660.00
9,1012,37,4,2455.30



========== CUSTOMER SEGMENTATION ==========
   customer_id  Recency  Frequency  Monetary              Segment
0         1001      136          3    649.45     Regular Customer
1         1002      103          2   1433.20     At Risk Customer
2         1004      134          1    601.80     At Risk Customer
3         1005        1          1   2218.25  High Value Customer
4         1006      300          1    887.30     At Risk Customer
5         1007       10          3   2467.70  High Value Customer
6         1008       98          3   1256.15     Regular Customer
7         1009       47          3   1100.55     Regular Customer
8         1010       83          3   1660.00     Regular Customer
9         1012       37          4   2455.30     Regular Customer

Customer Segment Summary:
Segment
Regular Customer             31
At Risk Customer             23
New / Occasional Customer    17
High Value Customer          12
Name: count, dtype: int64

Customer segmentation completed!

CUSTO

,customer_id,Recency,Frequency,Monetary,Segment
0,1001,136,3,649.45,Regular Customer
1,1002,103,2,1433.20,At Risk Customer
2,1004,134,1,601.80,At Risk Customer
3,1005,1,1,2218.25,High Value Customer
4,1006,300,1,887.30,At Risk Customer
5,1007,10,3,2467.70,High Value Customer
6,1008,98,3,1256.15,Regular Customer
7,1009,47,3,1100.55,Regular Customer
8,1010,83,3,1660.00,Regular Customer
9,1012,37,4,2455.30,Regular Customer



========== K-MEANS CLUSTERING ==========

Customer Cluster Results:
   customer_id  Recency  Frequency  Monetary  Cluster
0         1001      136          3    649.45        0
1         1002      103          2   1433.20        0
2         1004      134          1    601.80        1
3         1005        1          1   2218.25        0
4         1006      300          1    887.30        1
5         1007       10          3   2467.70        0
6         1008       98          3   1256.15        0
7         1009       47          3   1100.55        0
8         1010       83          3   1660.00        0
9         1012       37          4   2455.30        2

Cluster Summary:
Cluster
0    47
1    18
2    18
Name: count, dtype: int64

K-Means clustering completed successfully!

K-MEANS CLUSTERING COMPLETED!


,customer_id,Recency,Frequency,Monetary,Segment,Cluster
0,1001,136,3,649.45,Regular Customer,0
1,1002,103,2,1433.20,At Risk Customer,0
2,1004,134,1,601.80,At Risk Customer,1
3,1005,1,1,2218.25,High Value Customer,0
4,1006,300,1,887.30,At Risk Customer,1
5,1007,10,3,2467.70,High Value Customer,0
6,1008,98,3,1256.15,Regular Customer,0
7,1009,47,3,1100.55,Regular Customer,0
8,1010,83,3,1660.00,Regular Customer,0
9,1012,37,4,2455.30,Regular Customer,2



========== CUSTOMER LAPSE PREDICTION ==========

Customer Lapse Results:
   customer_id  Recency       Lapse_Status
0         1001      136  Inactive Customer
1         1002      103  Inactive Customer
2         1004      134  Inactive Customer
3         1005        1    Active Customer
4         1006      300  Inactive Customer
5         1007       10    Active Customer
6         1008       98  Inactive Customer
7         1009       47   At Risk Customer
8         1010       83   At Risk Customer
9         1012       37   At Risk Customer

Lapse Prediction Summary:
Lapse_Status
Inactive Customer    32
At Risk Customer     32
Active Customer      19
Name: count, dtype: int64

Customer lapse prediction completed!

CUSTOMER LAPSE PREDICTION COMPLETED!


,customer_id,Recency,Frequency,Monetary,Segment,Cluster,Lapse_Status
0,1001,136,3,649.45,Regular Customer,0,Inactive Customer
1,1002,103,2,1433.20,At Risk Customer,0,Inactive Customer
2,1004,134,1,601.80,At Risk Customer,1,Inactive Customer
3,1005,1,1,2218.25,High Value Customer,0,Active Customer
4,1006,300,1,887.30,At Risk Customer,1,Inactive Customer
5,1007,10,3,2467.70,High Value Customer,0,Active Customer
6,1008,98,3,1256.15,Regular Customer,0,Inactive Customer
7,1009,47,3,1100.55,Regular Customer,0,At Risk Customer
8,1010,83,3,1660.00,Regular Customer,0,At Risk Customer
9,1012,37,4,2455.30,Regular Customer,2,At Risk Customer



ALL PART B ANALYSIS COMPLETED SUCCESSFULLY!


# PART C – CUSTOMER SUPPORT TICKET INTELLIGENCE

## Aim
To analyse customer support tickets, identify customer issues, classify ticket categories, and assign priority levels.

## Objectives
1. Load customer support ticket data.
2. Identify and remove personal information (PII).
3. Classify support tickets based on their issues.
4. Assign priority levels to tickets.
5. Save the processed ticket information.

In [6]:

# =====================================================
# PART C - CUSTOMER SUPPORT TICKET INTELLIGENCE
# STEP 1 - RUN SUPPORT TICKET PROCESSING
# =====================================================

import sys
from pathlib import Path

# Connect notebook with src folder
PROJECT_DIR = Path.cwd()
SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Import support ticket module
import triage

# Run complete ticket processing
processed_tickets = triage.process_tickets()

# Display processed ticket information
print("\n========== FINAL TICKET RESULTS ==========")

display(
    processed_tickets[
        [
            "ticket_id",
            "Redacted_Message",
            "Category",
            "Priority",
            "Assigned_Team"
        ]
    ].head(10)
)

# Display category summary
print("\nTICKET CATEGORY SUMMARY")

display(processed_tickets["Category"].value_counts())

# Display priority summary
print("\nTICKET PRIORITY SUMMARY")

display(processed_tickets["Priority"].value_counts())

print("\nALL PART C OPERATIONS COMPLETED SUCCESSFULLY!")


INFO: Support tickets loaded successfully



========== SUPPORT TICKET INTELLIGENCE ==========

Processed Support Tickets:

 ticket_id                                       Redacted_Message           Category Priority        Assigned_Team
       501                   My order is delayed. Please help me.           Delivery      Low        Delivery Team
       502                          I received a damaged product.      Product Issue   Medium Product Support Team
       503                I want a refund for my cancelled order.             Refund   Medium          Refund Team
       504                 My payment was deducted twice. Urgent!            Payment     High         Payment Team
       505                          I received the wrong product.      Product Issue   Medium Product Support Team
       506                          I cannot login to my account.            Account      Low Account Support Team
       507                           My delivery has not arrived.           Delivery   Medium        Delivery Team


,ticket_id,Redacted_Message,Category,Priority,Assigned_Team
0,501,My order is delayed. Please help me.,Delivery,Low,Delivery Team
1,502,I received a damaged product.,Product Issue,Medium,Product Support Team
2,503,I want a refund for my cancelled order.,Refund,Medium,Refund Team
3,504,My payment was deducted twice. Urgent!,Payment,High,Payment Team
4,505,I received the wrong product.,Product Issue,Medium,Product Support Team
5,506,I cannot login to my account.,Account,Low,Account Support Team
6,507,My delivery has not arrived.,Delivery,Medium,Delivery Team
7,508,Please cancel my order.,Order Cancellation,Low,Order Support Team
8,509,I need help with my account.,Account,Low,Account Support Team
9,510,I was charged extra money. Please resolve imme...,Payment,High,Payment Team



TICKET CATEGORY SUMMARY


Category
Delivery              2
Product Issue         2
Payment               2
Account               2
Refund                1
Order Cancellation    1
Name: count, dtype: int64


TICKET PRIORITY SUMMARY


Priority
Low       4
Medium    4
High      2
Name: count, dtype: int64


ALL PART C OPERATIONS COMPLETED SUCCESSFULLY!


# PART D - POLICY BASED RAG AND REFUND SUPPORT AGENT

## Aim

To develop a policy-based customer support system using Retrieval-Augmented Generation (RAG) to retrieve refund information and assist customers with refund-related queries.

## Objectives

1. To load customer refund policy documents.
2. To divide policy documents into smaller chunks.
3. To retrieve relevant information using TF-IDF.
4. To calculate similarity between customer queries and policy documents.
5. To generate policy-based responses for customer queries.
6. To save refund support results for future reference.

In [7]:
# ==========================================================
# CUSTOMER INTELLIGENCE SUPPORT COPILOT
# PART D - POLICY BASED RAG AND REFUND SUPPORT AGENT
# ==========================================================

from pathlib import Path
import pandas as pd
import re
import logging

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ----------------------------------------------------------
# STEP 1: PROJECT SETUP
# ----------------------------------------------------------

BASE_DIR = Path.cwd()

POLICY_DIR = BASE_DIR / "data" / "policies"
OUTPUT_DIR = BASE_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

logger = logging.getLogger("RefundSupportAgent")


# ----------------------------------------------------------
# STEP 2: LOAD REFUND POLICY DOCUMENT
# ----------------------------------------------------------

def load_refund_policy():

    policy_file = POLICY_DIR / "refund_policy.txt"

    if not policy_file.exists():
        raise FileNotFoundError(
            "Refund policy file not found!"
        )

    with open(
        policy_file,
        "r",
        encoding="utf-8"
    ) as file:

        policy_text = file.read()

    if not policy_text.strip():
        raise ValueError(
            "Refund policy document is empty!"
        )

    logger.info(
        "Refund policy loaded successfully"
    )

    return policy_text


# ----------------------------------------------------------
# STEP 3: SPLIT POLICY INTO SMALLER CHUNKS
# ----------------------------------------------------------

def split_policy(policy_text):

    chunks = [
        paragraph.strip()
        for paragraph in re.split(
            r"\n\s*\n",
            policy_text
        )
        if paragraph.strip()
    ]

    if len(chunks) == 1:

        sentences = re.split(
            r"(?<=[.!?])\s+",
            chunks[0]
        )

        chunks = [
            sentence.strip()
            for sentence in sentences
            if sentence.strip()
        ]

    logger.info(
        "Policy divided into %d chunks",
        len(chunks)
    )

    return chunks


# ----------------------------------------------------------
# STEP 4: CREATE TF-IDF VECTORS
# ----------------------------------------------------------

def create_policy_vectors(policy_chunks):

    if not policy_chunks:
        raise ValueError(
            "No policy information available!"
        )

    vectorizer = TfidfVectorizer(
        stop_words="english"
    )

    policy_vectors = vectorizer.fit_transform(
        policy_chunks
    )

    logger.info(
        "TF-IDF vectors created successfully"
    )

    return vectorizer, policy_vectors


# ----------------------------------------------------------
# STEP 5: RETRIEVE RELEVANT POLICY INFORMATION
# ----------------------------------------------------------

def retrieve_policy(
    query,
    policy_chunks,
    vectorizer,
    policy_vectors
):

    query_vector = vectorizer.transform(
        [query]
    )

    similarity_scores = cosine_similarity(
        query_vector,
        policy_vectors
    )[0]

    best_index = similarity_scores.argmax()

    best_score = similarity_scores[best_index]

    if best_score <= 0:

        return (
            "No relevant policy information found.",
            0.0
        )

    return (
        policy_chunks[best_index],
        float(best_score)
    )


# ----------------------------------------------------------
# STEP 6: CREATE REFUND SUPPORT AGENT
# ----------------------------------------------------------

def refund_support_agent(
    customer_query,
    policy_chunks,
    vectorizer,
    policy_vectors
):

    print(
        "\n========== REFUND SUPPORT AGENT =========="
    )

    print("\nCustomer Query:")
    print(customer_query)

    retrieved_policy, score = retrieve_policy(
        customer_query,
        policy_chunks,
        vectorizer,
        policy_vectors
    )

    print("\nRetrieved Policy Information:")
    print(retrieved_policy)

    if score <= 0:

        response = (
            "Sorry, no relevant information was found "
            "in the refund policy. Please contact "
            "customer support."
        )

    else:

        response = (
            "Based on the refund policy, here is the "
            "relevant information:\n"
            + retrieved_policy
            + "\nPlease check the policy conditions "
            "before confirming refund eligibility."
        )

    print("\nAgent Response:")
    print(response)

    return response


# ----------------------------------------------------------
# STEP 7: PROCESS CUSTOMER REFUND QUERIES
# ----------------------------------------------------------

def process_refund_queries(
    policy_chunks,
    vectorizer,
    policy_vectors
):

    customer_queries = [

        "What is the refund policy?",

        "Can I get a refund for my cancelled order?",

        "How can I request a refund?",

        "What should I do if I receive a damaged product?",

        "How long does a refund take?"

    ]

    results = []

    print(
        "\n========== CUSTOMER REFUND QUERIES =========="
    )

    for query in customer_queries:

        retrieved_policy, score = retrieve_policy(
            query,
            policy_chunks,
            vectorizer,
            policy_vectors
        )

        if score > 0:

            response = (
                "Relevant policy information found: "
                + retrieved_policy
            )

        else:

            response = (
                "No relevant policy information found."
            )

        results.append({

            "Customer_Query": query,

            "Retrieved_Policy": retrieved_policy,

            "Similarity_Score": round(score, 3),

            "Agent_Response": response

        })

    results_df = pd.DataFrame(results)

    print("\nRefund Query Results:\n")

    print(
        results_df.to_string(index=False)
    )

    return results_df


# ----------------------------------------------------------
# STEP 8: SAVE REFUND AGENT RESULTS
# ----------------------------------------------------------

def save_refund_results(results_df):

    output_file = (
        OUTPUT_DIR / "refund_agent_results.csv"
    )

    results_df.to_csv(
        output_file,
        index=False
    )

    logger.info(
        "Refund agent results saved successfully"
    )

    print(
        "\nResults saved in:",
        output_file
    )


# ----------------------------------------------------------
# STEP 9: RUN COMPLETE REFUND SUPPORT AGENT
# ----------------------------------------------------------

if __name__ == "__main__":

    print("\n========== PART D STARTED ==========")

    # Load refund policy
    policy_text = load_refund_policy()

    # Split policy into chunks
    policy_chunks = split_policy(
        policy_text
    )

    # Create TF-IDF vectors
    vectorizer, policy_vectors = create_policy_vectors(
        policy_chunks
    )

    # Test one customer query
    test_query = (
        "I want to request a refund for my order."
    )

    refund_support_agent(
        test_query,
        policy_chunks,
        vectorizer,
        policy_vectors
    )

    # Process multiple customer queries
    results_df = process_refund_queries(
        policy_chunks,
        vectorizer,
        policy_vectors
    )

    # Save results
    save_refund_results(
        results_df
    )

    print(
        "\nALL PART D MODULES COMPLETED SUCCESSFULLY!"
    )

INFO: Refund policy loaded successfully
INFO: Policy divided into 9 chunks
INFO: TF-IDF vectors created successfully
INFO: Refund agent results saved successfully



========== PART D STARTED ==========

========== REFUND SUPPORT AGENT ==========

Customer Query:
I want to request a refund for my order.

Retrieved Policy Information:
1. REFUND ELIGIBILITY
Customers can request a refund if they receive a damaged product, wrong product, or an eligible cancelled order.

Agent Response:
Based on the refund policy, here is the relevant information:
1. REFUND ELIGIBILITY
Customers can request a refund if they receive a damaged product, wrong product, or an eligible cancelled order.
Please check the policy conditions before confirming refund eligibility.

========== CUSTOMER REFUND QUERIES ==========

Refund Query Results:

                                  Customer_Query                                                                                                                                          Retrieved_Policy  Similarity_Score                                                                                                                    